# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fatima12aa/fa-ml/blob/main/work/notebooks/capstone.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

*The research question and the decision it supports.*

## Question

**Research question:** Can a machine learning model, using verified search performance
signals (click-through rate and average search position), produce a more precise ranked
queue of pages requiring review than a hand-written rule using the same signals?

**The decision this supports:** Which pages a FlyRank content reviewer should prioritize
reviewing first, out of a large page inventory, when they have limited time and cannot
review every page individually.

**Unit of analysis:** One row = one content page, evaluated using its February 2026
performance signals (impressions, CTR, average position).

**The action a human takes:** A reviewer opens the ranked queue and works through it in
order, starting a title/meta review on the highest-priority flagged pages first.

**Cost of a wrong call:** Two-directional. Flagging a page that didn't need review wastes
reviewer time but causes no lasting harm. Missing a page that genuinely needed review risks
that page continuing to lose traffic silently — a more expensive mistake, since visibility
lost isn't always easy to recover (echoing the FlyRank paper's own finding on the value of
timely refresh).

**Why ML, not just a fixed rule:** Our Week 4 hand-written rule (a binary gate: good position
AND low CTR) achieved Precision@20 = 0.15, Precision@50 = 0.22. Restricting a Random Forest
model to the same two verified signals — but letting it learn graded relationships rather
than a strict AND-gate — improved this to Precision@20 = 0.25-0.35, Precision@50 = 0.16-0.32
(varying by validation split). This is genuine, evidence-based justification for using ML
here: the pattern (how CTR and position jointly relate to decline risk) was real but too
graded/nuanced for a simple hand-written threshold to fully capture. The model can output a continuous probability, while the hand written rule was just a binary gate giving hard coded 0 or 1.

## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

## Data

**Release used:** FlyRank's internship data warehouse (`FlyRank/internship-warehouse` on
Hugging Face), build `flyrank_pseudonymized_warehouse_release_v20260703`.

**Tables used:**
- `fact_content_daily_performance` — daily, page-level GSC performance signals (impressions,
  clicks, average position), filtered to February and March 2026.
- `dim_content` — static page-level characteristics (word count, search volume), used during
  exploratory feature testing.

**Date windows:** February 2026 (feature window) and March 2026 (label/outcome window) —
a mid-panel period chosen specifically to avoid the final month of the dataset (June 2026),
which would have no genuine "future" period available for safe label construction.

**What we excluded, and why (public-safe):**
- `trend_direction` / `trend_pct`: pre-built columns directly derived from the same
  month-over-month comparison we needed to predict — using them would be leakage (feeding
  the model the answer disguised as a feature).
- `march_impressions` and any March-window signal: excluded from features entirely, since
  March defines our label; only used to construct the label itself, never as model input.
- `provider_used` / `model_used`: 71.5% missing, with risk of confounding (correlation with
  decline may reflect *when* a tool was adopted, not genuine content quality).
- `ai_traffic_pct`: 93.6% of rows were exactly zero — too sparse to support reliable
  page-level claims.
- Rows lacking genuine GSC tracking (`gsc_data_available` = FALSE): excluded via SQL filter,
  since these are zero-filled placeholders, not real measurements.
- All client and content identifiers are pseudonymized hashes (`client_hash_id`,
  `content_hash_id`) — used only for joining and grouping (e.g. client-holdout validation),
  never as model features, and no raw client names, URLs, or private queries appear anywhere
  in this work.

## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

## Methodology

**Assumptions:**
- A page is labeled "declining" if its March 2026 impressions fall below 80% of its
  February 2026 impressions (a genuine 20%+ month-over-month drop). This threshold is a
  policy choice, not a universal rule — a stricter or looser cutoff would flag a different
  set of pages.
- We assume February performance data is representative enough to serve as a fair prediction
  window for March outcomes; we do not claim this generalizes to arbitrary other month pairs
  without re-validation.

**Label definition (one sentence):** `is_declining = 1` if a page's March 2026 impressions
are less than 80% of its February 2026 impressions, else `0`.

**Features used (final model):**
- `feb_ctr` — February click-through rate (clicks ÷ impressions)
- `feb_avg_position` — average February search ranking position

**Features deliberately left out of the final model** (tested earlier, found not to help):
`word_count`, `search_volume`, `feb_impressions` — an earlier 5-feature version of this model
gave `word_count` the highest feature importance, yet performed no better than the baseline
rule on a held-out test set. This is a caution about feature importance: high importance
during training does not guarantee real predictive value on unseen data. Restricting to only
the two features we had independently verified as meaningful (Section on Signal Audit, below)
produced a genuine improvement instead.

**Signal audit (verification before building the rule):**
- CTR vs. position: CONFIRMED. Pages ranked in the top 10 positions showed meaningfully
  higher CTR (mean 0.0061) than pages ranked 11-30 (0.0030) or 30+ (0.0024), across a large
  sample (n=151,956).
- Volume vs. decline: OPPOSITE of our original hypothesis. Higher-traffic pages showed LOWER
  decline rates (15.7%) than lower-traffic pages (22.4%) — a clean, monotonic pattern
  (n=134,238). This finding reshaped how volume was used: not as a standalone risk signal,
  but as a way to weigh the value of fixing an already-declining page.

**Baseline:** A transparent, hand-written rule: flag a page if its average February position
is ≤10 AND its February CTR is below 0.006083 (the mean CTR of top-10-position pages). This
is a fair comparison because it uses the exact same two verified signals as the final model,
differing only in that it applies a strict binary threshold rather than a learned, graded
relationship.

**Model:** Random Forest Classifier, trained on the two verified features only
(`feb_ctr`, `feb_avg_position`).

**Validation design:** Grouped by `client_hash_id` using `GroupShuffleSplit` (25% of clients
held out entirely for testing) — not a plain random row-level split. This matters because
pages from the same client may share client-specific patterns; a naive random split lets the
model partially "memorize" a client seen in training and inflates test performance. We
directly verified zero client overlap between train and test sets before trusting any result.

**Leakage checks performed:**
- Confirmed `is_declining` and `march_impressions` never appear among model features
  (verified programmatically via assertion, not just asserted in prose).
- Confirmed all dynamic features (`feb_ctr`, `feb_avg_position`) are built from SQL queries
  explicitly restricted to the February date range, with no March dates included.
- Confirmed pseudonymous IDs (`client_hash_id`, `content_hash_id`) are used only for joining
  and grouping, never as model features.

## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

## 5. Limitations

*What this work cannot claim.*

## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.